# 10 · OranjeCart — Two-Table Take-Home (Bronze → Silver → Gold)

A take-home style project with **two related dirty tables**: a customer list from a CRM export
and an order log from the order system (synthetic practice data, an online shop delivering to
5 Dutch cities). The finance team doesn't trust either file: *"the numbers don't add up, the
customer list looks inflated, and some orders don't seem to belong to anyone."*

The job: clean both tables into Silver, check how they relate, and build three finance-ready
Gold tables. No task list was given — only business requirements, so every step below starts
from a decision I had to make and defend.

What this project drills on top of the previous ones:

- cleaning **two tables that must join** — so deduplication mistakes would multiply rows later
- **recovering** corrupted values from other columns instead of dropping rows
- a row-by-row **reconciliation (trust check)**: does `total_amount` really equal `qty × price`?
- **anti joins** in both directions (orders without a customer / customers without orders)
- a Gold summary that must include **customers with zero orders** — the classic
  "filter before or after the outer join" trap


## 1 · Read raw (Bronze)

Everything is read **as string** first. Types come later, after I've seen what's actually
in the columns — casting blind is how you lose data silently.


In [ ]:
nl_raw_customers_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/oranjecart_customers.csv")
)
nl_raw_customers_df.display()


In [ ]:
nl_raw_orders_df = (
    spark.read.format("csv")
    .option("header", True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/oranjecart_orders.csv")
)
nl_raw_orders_df.display()


### Grain contract — prove it, don't assume it

Before touching anything I state what one row should be and prove it with counts.

**customers** came back as: 57 rows, 52 distinct `customer_id`, 0 null ids, 54 distinct
full rows. So the CRM's "unique customers" claim is false three ways: 3 rows are exact
copies (57 → 54), and 2 more ids repeat with rows that *look* identical but aren't
byte-identical — those turned out to be case/whitespace variants (more below).


In [ ]:
from pyspark.sql.functions import col

# grain control — customers
print(nl_raw_customers_df.select("customer_id").count())            # 57 rows
print(nl_raw_customers_df.select("customer_id").distinct().count()) # 52 unique customers
print(nl_raw_customers_df.filter(col("customer_id").isNull()).count())  # 0 null keys
print(nl_raw_customers_df.count())                                  # 57
print(nl_raw_customers_df.distinct().count())                       # 54 -> 3 exact duplicate rows


**orders**: 136 rows → 132 distinct full rows (4 exact duplicate rows), and after
dedup `order_id` is unique (132 = 132). Grain: **one row = one order**.


## 2 · customers_silver

### Step 1 — drop the exact copies

Full-row `dropDuplicates()` is safe at this point: whichever copy survives, it's the same row.


In [ ]:
nl_raw_customers_df_dedup = nl_raw_customers_df.dropDuplicates()
nl_raw_customers_df_dedup.count()   # 57 -> 54


### Step 2 — find out why 2 ids still repeat

54 rows but only 52 unique ids — so two customers still have two rows each. I pulled them
up instead of deleting blind:


In [ ]:
nl_raw_customers_df_dedup.groupBy("customer_id").count().filter(col("count") > 1).display()
# -> C010 and C023, 2 rows each

nl_raw_customers_df_dedup.filter(col("customer_id").isin("C010", "C023")).display()


Side by side, the pairs differ only by **case and hidden whitespace**
(`IRIS HENDRIKS` vs `Iris Hendriks`, an email with a trailing space). Same person, same data —
just formatting noise. That's why full-row dedup didn't catch them: *looks identical ≠ is identical*.

The order of operations matters here: **normalize first, then dedup again.** After trimming
and fixing case, these pairs become exact copies and `dropDuplicates()` removes them safely —
no risk of keeping the dirty version, because both versions are now the clean one.
Deduplicating on `customer_id` alone instead would keep an arbitrary row, possibly the
`IRIS HENDRIKS` one.


In [ ]:
from pyspark.sql.functions import trim, initcap, lower

nl_raw_customers_df_dedup = nl_raw_customers_df_dedup.withColumns({
    "customer_id" : trim(col("customer_id")),
    "full_name"   : trim(initcap(col("full_name"))),
    "email"       : trim(lower(col("email"))),
    "city"        : trim(initcap(col("city")))
}).dropDuplicates()

print(nl_raw_customers_df_dedup.count())                                  # 52
print(nl_raw_customers_df_dedup.select("customer_id").distinct().count()) # 52 -> grain proven


52 rows = 52 distinct ids. **Grain proven: one row = one customer.**

### Step 3 — placeholder emails → real NULL

One detail bit me here: my own normalization had already lowercased the placeholders,
so I had to search for `n/a` / `unknown`, not `N/A` / `UNKNOWN`. Measured first: 11 rows.


In [ ]:
nl_raw_customers_df_dedup.filter(col("email").isin("n/a", "error", "unknown")).display()  # 11 rows


In [ ]:
from pyspark.sql.functions import when

nl_raw_customers_df_dedup = (
    nl_raw_customers_df_dedup
    .withColumn("email", when(col("email").isin("n/a", "unknown", "error"), None)
                .otherwise(col("email")))
)


### Step 4 — signup_date: three formats, one date column

The column mixed `2024-04-15`, `24/7/2023` and `Sep 3, 2024`. One `try_to_date` per format
inside `coalesce` — each try returns null if the pattern doesn't match, and coalesce keeps
the first success. Note the single-letter `d` / `M`: they accept 1 *or* 2 digits, so `3` and
`03` both parse. `dd` would silently null every single-digit day.


In [ ]:
from pyspark.sql.functions import coalesce, try_to_date

nl_normalized_customers_df = nl_raw_customers_df_dedup.withColumns({
    "signup_date": coalesce(
        try_to_date("signup_date", "yyyy-MM-dd"),
        try_to_date("signup_date", "d/M/yyyy"),
        try_to_date("signup_date", "MMM d, yyyy")
    )
})


In [ ]:
from pyspark.sql.functions import min, max

# proof: the parse lost nothing, and the range makes sense
print(nl_normalized_customers_df.filter(col("signup_date").isNull()).count())   # 0 unparsed
nl_normalized_customers_df.select(min("signup_date"), max("signup_date")).display()
# 2023-02-08 -> 2024-12-20, all in the past — sane


## 3 · orders_silver

### Step 1 — systematic placeholder scan

Instead of eyeballing, I scanned every column for the known junk tokens. Only two columns
had them: `quantity` (one `N/A`) and `unit_price` (three `ERROR`).


In [ ]:
for c in ["order_id", "customer_id", "quantity", "unit_price", "total_amount", "status", "order_date"]:
    n = nl_raw_orders_df_dedup.filter(lower(col(c)).isin("n/a", "error", "unknown")).count()
    print(c, "->", n)


In [ ]:
nl_orders_normalized = nl_raw_orders_df_dedup.withColumns({
    "quantity"   : when(col("quantity").isin("N/A"), None).otherwise(col("quantity")),
    "unit_price" : when(col("unit_price").isin("ERROR"), None).otherwise(col("unit_price"))
})


### Step 2 — currency text before the cast

`unit_price` held three different notations for the same thing: `26.50`, `€48.00`,
`75,00 EUR`. A letter-hunt (`rlike("[A-Za-z]")`) surfaced 17 rows — and this is where
"letters = delete" would have been a disaster: 14 of the 17 were **real prices** with an
`EUR` suffix, only 3 were `ERROR` junk. Letters mean *inspect*, not *drop*.

So: strip symbols, unify the decimal separator, then cast. Note that `replace()` treats bare
strings as **column names** — literal text has to be wrapped in `lit()` (learned that from an
`INVALID_ATTRIBUTE_NAME` error).


In [ ]:
from pyspark.sql.functions import replace, lit

nl_orders_normalized = nl_orders_normalized.withColumn(
    "unit_price",
    replace(replace(replace("unit_price", lit("€"), lit("")), lit("EUR"), lit("")), lit(","), lit("."))
)


### Step 3 — real types

Dates had two formats (`ISO` + `d/M/yyyy`). Money as `decimal(10,2)` — never float for money.


In [ ]:
nl_normalized_order_df = nl_orders_normalized.withColumns({
    "order_date"   : coalesce(
        try_to_date("order_date", "d/M/yyyy"),
        try_to_date("order_date", "yyyy-MM-dd")
    ),
    "quantity"     : col("quantity").cast("int"),
    "unit_price"   : trim(col("unit_price")).cast("decimal(10,2)"),
    "total_amount" : col("total_amount").cast("decimal(10,2)")
})

print(nl_normalized_order_df.filter(col("order_date").isNull()).count())  # 0 unparsed


### Step 4 — impossible quantities: recover, don't drop

Four rows had `quantity` of 0 or negative — and one more was null from the `N/A`.
A zero-quantity *Delivered* order with €339 collected makes no sense, so something was
broken. Before deleting I checked whether the row could tell me the true value:


In [ ]:
(nl_normalized_order_df.filter(col("quantity") <= 0)
    .withColumn("implied_qty", col("total_amount") / col("unit_price"))
    .display())
# 184.0/92.0 = 2, 339.0/113.0 = 3, 244.5/81.5 = 3, 460.0/92.0 = 5 — all clean integers


Every division came out a **clean integer**. If these rows were random garbage the
ratios would be junk — clean integers mean the order was real, `total_amount` was computed
from a real quantity, and only the `quantity` *field* got corrupted somewhere. So the row
isn't broken; one cell is, and the other two cells contain enough information to rebuild it.

Important: the repair is applied **only to the broken rows** (`when/otherwise`). Recomputing
quantity for *all* rows would assume `total = qty × price` holds everywhere — which I hadn't
verified yet (that's the trust check below), and it would have destroyed the very evidence
that check needs.


In [ ]:
nl_normalized_order_df = nl_normalized_order_df.withColumn(
    "quantity",
    when((col("quantity") <= 0) | col("quantity").isNull(),
         col("total_amount") / col("unit_price"))
    .otherwise(col("quantity"))
    .cast("int")
)

print(nl_normalized_order_df.filter((col("quantity") <= 0) | col("quantity").isNull()).count())  # 0


### Step 5 — same trick in the other directions

3 null `unit_price` (the `ERROR` rows) recovered from `total ÷ qty`, and 5 missing
`total_amount` rebuilt from `qty × price`. These 8 reconstructed rows are flagged in the
decision log — they now satisfy `total = qty × price` **by construction**, so they can't
count as evidence in the trust check.


In [ ]:
nl_normalized_order_df = nl_normalized_order_df.withColumns({
    "unit_price"   : when(col("unit_price").isNull(), col("total_amount") / col("quantity"))
                     .otherwise(col("unit_price")).cast("decimal(10,2)"),
    "total_amount" : when(col("total_amount").isNull(), col("quantity") * col("unit_price"))
                     .otherwise(col("total_amount"))
})

print(nl_normalized_order_df.filter(col("unit_price").isNull()).count())    # 0
print(nl_normalized_order_df.filter(col("total_amount").isNull()).count())  # 0


### Step 6 — null customer_id: keep the money

3 orders have no `customer_id` at all (combined value **€306.50**). The money is real, the
customer reference is broken — so the rows stay for total revenue, but they can't appear in
any per-customer or per-city breakdown. That difference between "total" and "sum of the
breakdown" is documented, not hidden.


## 4 · Trust check — does `total_amount` = `qty × price`?

Finance's actual complaint. Row-by-row reconciliation:


In [ ]:
print(nl_normalized_order_df.filter(col("total_amount") == col("unit_price") * col("quantity")).count())  # 120
print(nl_normalized_order_df.filter(col("total_amount") != col("unit_price") * col("quantity")).count())  # 12


In [ ]:
# don't "fix" a mismatch before understanding it — look for a pattern
(nl_normalized_order_df
    .filter(col("total_amount") != col("quantity") * col("unit_price"))
    .withColumn("ratio", col("total_amount") / (col("quantity") * col("unit_price")))
    .display())


**All 12 mismatches have a ratio of ≈ 0.90** (exact 0.90 on most rows, 0.9002–0.9007 on the
rest — that's just the 2-decimal rounding of the stored totals). And all 12 are **Delivered**.

Twelve rows landing on exactly 90%, all in one status, is not random data entry error — it's
a systematic pattern, consistent with a **10% discount** applied to those orders. I'd confirm
the exact business rule with the source team, but the revenue decision doesn't depend on the
name of the rule: `total_amount` is the money actually collected, so **revenue is based on
`total_amount`**. Recomputing it as `qty × price` would report ~10% money that never arrived
on those orders — which is probably the exact discrepancy finance was complaining about.


## 5 · Relationship checks — both directions, before any reporting join

**(a) Orders with no matching customer.** An anti join from orders against customers:
9 orders — the 3 with null ids plus 6 carrying ids that simply don't exist in the CRM
(`C901`, `C902`, `C903`, `C915`). Money kept in totals, excluded from breakdowns —
same decision as the null-id rows.


In [ ]:
orphan_orders = nl_normalized_order_df.join(nl_normalized_customers_df, "customer_id", "left_anti")
print(orphan_orders.count())   # 9  (3 null ids + 6 unknown ids)
orphan_orders.display()


**(b) Customers who never ordered.** Same weapon, opposite direction — customers on the
left this time. 8 people. They must still show up in the Gold customer summary with zeros.


In [ ]:
never_ordered = nl_normalized_customers_df.join(nl_normalized_order_df, "customer_id", "left_anti")
print(never_ordered.count())   # 8
never_ordered.display()


In [ ]:
# same result the classic SQL way: LEFT JOIN ... WHERE right side IS NULL
from pyspark.sql.functions import expr

never_ordered_v2 = (
    nl_normalized_customers_df.alias("c")
    .join(nl_normalized_order_df.alias("o"), expr("o.customer_id == c.customer_id"), "left")
    .filter(col("o.customer_id").isNull())
)
print(never_ordered_v2.count())  # 8 — matches the anti join


## 6 · Gold layer

### The reporting join — with a fan-out proof

Orders are the subject (revenue lives there), customer attributes get attached with a left
join. Because customers were deduplicated to one row per id first, the join **must not**
change the row count — and it doesn't: 132 before, 132 after. If I had skipped the dedup,
the two repeated customers would have doubled their orders here and quietly inflated revenue.


In [ ]:
joined_df = nl_normalized_order_df.join(nl_normalized_customers_df, on="customer_id", how="left")

print(nl_normalized_order_df.count())  # 132
print(joined_df.count())               # 132 -> no fan-out


### Which statuses count as revenue?

Statuses after normalization: `Delivered`, `Shipped`, `In Transit`, `Cancelled`, `Returned`.
My call: **Delivered + Shipped** count as revenue (payment collected, order fulfilled or on
its final leg); `Cancelled` and `Returned` don't — returned money goes back, same reasoning
as my UK retail project; `In Transit` left out pending a business definition. Whatever the
final ruling, it's one `isin` whitelist away — and a whitelist beats a chain of `!=`
exclusions, which I proved to myself the hard way (one wrong `|` in an exclusion chain and
the filter silently kept *everything*; a whitelist can't fail that way).

### G1 · monthly revenue per city

One trap here: grouping by month **name** merges June 2024 with June 2025. `yyyy-MM` keeps
the years apart. Grain: one row = one month × city. The 9 unattributable orders have no
city, so they're excluded here by design.


In [ ]:
from pyspark.sql.functions import date_format, sum

gold_monthly_city_revenue = (joined_df
    .filter(lower(col("status")).isin("delivered", "shipped"))
    .filter(col("city").isNotNull())
    .withColumn("month", date_format(col("order_date"), "yyyy-MM"))
    .groupBy("month", "city")
    .agg(sum("total_amount").alias("revenue"))
    .orderBy("month", "city"))

gold_monthly_city_revenue.display()


In [ ]:
# supporting view — revenue per city overall
revenue_by_city_df = (joined_df
    .filter(lower(col("status")).isin("delivered", "shipped"))
    .filter(col("city").isNotNull())
    .groupBy("city")
    .agg(sum("total_amount").alias("total_revenue"))
    .orderBy(col("total_revenue").desc()))

revenue_by_city_df.display()


### G2 · top 10 customers by revenue

Grain: one row = one customer, top 10 by revenue. Null-id orders can't participate — no
customer to attribute the money to.


In [ ]:
from pyspark.sql.functions import count

gold_top10_customers = (joined_df
    .filter(lower(col("status")).isin("delivered", "shipped"))
    .filter(col("customer_id").isNotNull())
    .groupBy("customer_id", "full_name")
    .agg(sum("total_amount").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
    .limit(10))

gold_top10_customers.display()


### G3 · 2025 summary for ALL customers — zeros included

Finance wants every customer in the list, including the 8 who never ordered and the ones
whose only orders were in 2024. That splits the work in two:

1. the **numbers** can only come from orders → filter (2025 + revenue statuses) and
   aggregate per customer
2. the **complete list** can only come from customers → customers on the left, summary
   left-joined on

The filter has to happen **before** the join. Filtering `year == 2025` after a left join
would drop exactly the customers this table exists to show — their order columns are null,
and null never passes a comparison.


In [ ]:
from pyspark.sql.functions import year

summary_2025 = (nl_normalized_order_df
    .filter((year(col("order_date")) == 2025)
            & lower(col("status")).isin("delivered", "shipped"))
    .groupBy("customer_id")
    .agg(count("order_id").alias("orders_2025"),
         sum("total_amount").alias("revenue_2025")))


In [ ]:
gold_customer_2025_summary = (nl_normalized_customers_df
    .join(summary_2025, "customer_id", "left")
    .fillna(0, subset=["orders_2025", "revenue_2025"]))

print(gold_customer_2025_summary.count())   # 52 — one row per customer, nobody lost
gold_customer_2025_summary.display()


## 7 · Idempotent writes + final checks

Everything written with `mode("overwrite")` — a rerun rebuilds the same tables instead of
appending duplicates. A rerun should be boring.


In [ ]:
nl_normalized_customers_df.write.mode("overwrite").saveAsTable("dev.spark_db.customers_silver")
nl_normalized_order_df.write.mode("overwrite").saveAsTable("dev.spark_db.orders_silver")

gold_monthly_city_revenue.write.mode("overwrite").saveAsTable("dev.spark_db.gold_monthly_city_revenue")
gold_top10_customers.write.mode("overwrite").saveAsTable("dev.spark_db.gold_top10_customers")
gold_customer_2025_summary.write.mode("overwrite").saveAsTable("dev.spark_db.gold_customer_2025_summary")


In [ ]:
# FINAL CHECKS
print(spark.table("dev.spark_db.customers_silver").count())                                   # 52
print(spark.table("dev.spark_db.customers_silver").select("customer_id").distinct().count())  # 52 (grain)
print(spark.table("dev.spark_db.orders_silver").count())                                      # 132
print(spark.table("dev.spark_db.orders_silver").select("order_id").distinct().count())        # 132 (grain)
print(spark.table("dev.spark_db.gold_customer_2025_summary").count())                         # 52
print(spark.table("dev.spark_db.orders_silver").filter(col("total_amount") < 0).count())      # 0 (business rule)


## 8 · Decision log

**customers_silver**
- 57 rows → 54 (3 exact duplicates) → text normalized (trim; emails lower, names/cities
  initcap) → full-row dedup caught 2 more case/whitespace near-duplicates (C010, C023).
  Final: 52 rows = 52 distinct ids. Grain: one row = one customer.
- 11 placeholder emails (`n/a`, `unknown`) → NULL.
- `signup_date`: 3 formats → `coalesce(try_to_date × 3)`, 0 unparsed.
  Range 2023-02-08 → 2024-12-20.

**orders_silver**
- 136 rows → 132 (4 exact duplicates); `order_id` unique. Grain: one row = one order.
- 5 corrupted quantities (2 zero, 2 negative, 1 N/A): `total ÷ price` gave clean integers
  for every one → **recovered instead of dropped**. No revenue lost.
- 3 null unit prices recovered from `total ÷ qty`; 5 missing totals from `qty × price`.
  These 8 reconstructed rows are excluded as evidence in the trust check.
- `order_date`: 2 formats → 0 unparsed. Range 2024-06-05 → 2025-06-27.
- 3 orders with null `customer_id` (€306.50): kept for total revenue, excluded from
  per-customer/per-city breakdowns.

**Trust check**
- 12 of 132 rows fail `total = qty × price`. All ratios ≈ 0.90 (within rounding), all
  Delivered → systematic, consistent with a 10% discount; would confirm the rule with the
  business. Decision: **revenue = total_amount** (money actually collected).

**Relationships**
- 9 orphan orders (3 null + 6 unknown ids): kept in totals, out of breakdowns.
- 8 customers never ordered: present in G3 with zeros.

**Gold**
- Revenue statuses: Delivered + Shipped; Cancelled/Returned excluded (refunded money is
  not revenue); In Transit pending business confirmation.
- G1 grain: month (`yyyy-MM`) × city — month *names* alone would merge 2024 and 2025.
- G3: filter before the join, customers LEFT JOIN summary, `fillna(0)` → 52 rows.
- Reporting join preserved 132 rows — no fan-out, because customers were deduplicated first.

**Idempotency**: all writes `overwrite`; reruns converge to the same state.


## Key takeaways

- **Normalize before you deduplicate.** Case/whitespace variants of the same record survive
  a full-row dedup; after normalization they collapse into exact copies and dedup removes
  them safely — no arbitrary "which copy wins" problem.
- **Letters in a numeric column mean *inspect*, not *delete*.** 14 of 17 flagged prices were
  real values with an `EUR` suffix. Nulling everything with a letter would have silently
  destroyed revenue.
- **Recover before you drop.** Impossible quantities whose `total ÷ price` divides into clean
  integers aren't broken rows — they're broken *cells* with the truth stored one column over.
- **Repair only the sick rows.** Applying the recovery formula to every row would have
  manufactured a 100% pass rate on the trust check and erased the discount pattern.
- **A mismatch with a constant ratio is a business rule, not an error.** 12 rows at exactly
  0.90, all Delivered = a discount, and the collected amount is what revenue means.
- **The subject of the question picks the join direction.** Orders without customers and
  customers without orders are two different anti joins — 9 and 8 are answers to two
  different questions.
- **Filter before an outer join when the filter touches the nullable side.** Otherwise the
  zero-order customers the report exists for are silently dropped.
- **Prove the join did no harm.** 132 rows in, 132 out — the one-line check that catches
  fan-out before it inflates a report.
